# basic CytoVI workflow

In [ ]:
import cytovi
import readfcs
import scanpy as sc

In [ ]:
# read data
adata_batch1 = readfcs.read('../data/raw/Spectral flow/Nunez/For Chiquito/Raw_100000/batch1')
adata_batch1.layers['raw'] = adata_batch1.X.copy()

adata_batch2 = readfcs.read('../data/raw/Spectral flow/Nunez/For Chiquito/Raw_100000/batch2')
adata_batch2.layers['raw'] = adata_batch2.X.copy()

In [ ]:
# preprocess
cytovi.pp.arcsinh(adata_batch1, global_scaling_factor=2000)
cytovi.pp.scale(adata_batch1)

cytovi.pp.arcsinh(adata_batch2, global_scaling_factor=2000)
cytovi.pp.scale(adata_batch2)

In [ ]:
# merge batches
adata = cytovi.pp.merge_batches([adata_batch1, adata_batch2])

In [ ]:
# do some plotting
cytovi.pl.histogram(adata, marker = 'all', groupby='batch', layer_key='transformed')
cytovi.pl.biaxial(adata, marker_x = 'CD3', marker_y = 'CD4', groupby='batch', layer_key='transformed')

In [ ]:
# train model
cytovi.CytoVI.setup_anndata(adata, layer='scaled', batch_key='batch')
model = cytovi.CytoVI(adata)
model.train()

In [ ]:
# compute umap and cluster CytoVI latent space
adata.obsm["X_CytoVI"] = model.get_latent_representation()
sc.pp.neighbors(adata, use_rep="X_CytoVI")
sc.tl.umap(adata)
sc.tl.leiden(adata, key_added="leiden_CytoVI")

In [ ]:
# plot data
sc.pl.umap(adata, color=["leiden_CytoVI", "batch"])
sc.pl.matrixplot(adata, var_names=adata.var_names, groupby="leiden_CytoVI")